In [ ]:
uv pip install torch==2.8.0 torchvision==0.23.0 nvidia-modelopt git+https://github.com/ultralytics/ultralytics@qat-nvidia

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import torch.nn.functional as F
import math
from ultralytics import YOLO
from ultralytics.utils.torch_utils import ModelEMA
from ultralytics.utils import LOGGER
import modelopt.torch.prune as mtp

# ── Configuration ──
TEACHER_PATH = '/content/drive/MyDrive/PBL5_Final_Results/v8m_eb_observer5/weights/best.pt'
DATA_YAML = '/content/drive/MyDrive/Work3.yolov8/data.yaml'
STUDENT_BASE = "yolov8n.pt"
PRUNE_FLOPS_TARGET = "70%"
KD_ALPHA = 0.5
FASTNAS_MAX_ITER = 20

TRAIN_EPOCHS = 70
TRAIN_IMGSZ = 640
TRAIN_BATCH = 32
TRAIN_LR0 = 0.0001
OUTPUT_PROJECT = '/content/drive/MyDrive/PBL5_Final_Results'
OUTPUT_NAME = 'v8n_Pruned_KD_v2'

# ── GPU check ──
assert torch.cuda.is_available(), "CUDA required"
gpu_props = torch.cuda.get_device_properties(0)
gpu_mem: float = getattr(gpu_props, 'total_memory', getattr(gpu_props, 'total_mem', 0)) / 1e9
print(f"[DEBUG] PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
print(f"[DEBUG] GPU 0: {gpu_props.name} ({gpu_mem:.1f} GB)")

# ── Load teacher once (frozen, eval-only) ──
model = YOLO(STUDENT_BASE)

teacher = YOLO(TEACHER_PATH).model.eval()
for p in teacher.parameters():
    p.requires_grad = False
teacher_params: int = sum(p.numel() for p in teacher.parameters())
print(f"[DEBUG] Teacher loaded: {teacher_params:,} params (frozen)")
print(f"[DEBUG] Student base: {STUDENT_BASE}")
print(f"[DEBUG] KD alpha: {KD_ALPHA} | Prune target: {PRUNE_FLOPS_TARGET}")

In [ ]:
class PrunedTrainer(model.task_map[model.task]["trainer"]):
    """FastNAS physical pruning + KD fine-tuning trainer.

    Fixes over Untitled14:
      1. No double forward pass — reuses preds from training loop.
      2. Targets preds[1] spatial logits (P3/P4/P5 Detect head outputs).
      3. Spatial interpolation for H/W mismatch after pruning.
      4. Convex combination: (1-α)*L_det + α*L_kd instead of additive.
    """

    def _setup_train(self):
        super()._setup_train()

        self._teacher = teacher.to(self.device)
        self._kd_alpha: float = KD_ALPHA

        def collect_func(batch):
            return self.preprocess_batch(batch)["img"]

        def score_func(m):
            m.eval()
            self.validator.args.save = False
            self.validator.args.plots = False
            self.validator.args.verbose = False
            self.validator.args.data = DATA_YAML
            metrics = self.validator(model=m)
            return metrics["fitness"]

        self.model.is_fused = lambda: True
        LOGGER.info("--- FASTNAS: searching optimal subnetwork ---")

        self.model, prune_res = mtp.prune(
            model=self.model,
            mode="fastnas",
            constraints={"flops": PRUNE_FLOPS_TARGET},
            dummy_input=torch.randn(1, 3, self.args.imgsz, self.args.imgsz).to(self.device),
            config={
                "score_func": score_func,
                "checkpoint": "modelopt_fastnas_search_checkpoint.pth",
                "data_loader": self.train_loader,
                "collect_func": collect_func,
                "max_iter_data_loader": FASTNAS_MAX_ITER,
            },
        )

        self.model.to(self.device)
        self.ema = ModelEMA(self.model)

        weight_decay = self.args.weight_decay * self.batch_size * self.accumulate / self.args.nbs
        iterations = math.ceil(
            len(self.train_loader.dataset) / max(self.batch_size, self.args.nbs)
        ) * self.epochs
        self.optimizer = self.build_optimizer(
            model=self.model,
            name=self.args.optimizer,
            lr=self.args.lr0,
            momentum=self.args.momentum,
            decay=weight_decay,
            iterations=iterations,
        )
        self._setup_scheduler()
        LOGGER.info(f"--- PRUNED SUCCESSFULLY: {prune_res} ---")

    def loss(self, batch, preds=None):
        """KD-enhanced detection loss with 4 fixes."""

        # FIX 1: Reuse preds from training loop — no redundant forward pass.
        # The base trainer already did preds = self.model(batch["img"]) and
        # passes it here. Only compute defensively if preds is None.
        if preds is None:
            preds = self.model(batch["img"])

        # Detection loss (box + cls + dfl). Passing preds through avoids
        # super() calling self.model() again internally.
        student_loss, loss_items = super().loss(batch, preds)

        # Teacher forward (frozen, no gradient)
        with torch.no_grad():
            t_preds = self._teacher(batch["img"])

        # FIX 2: Target the correct spatial logits from Detect heads.
        # In YOLOv8 training mode preds = (decoded, [P3, P4, P5]).
        # In eval mode (teacher) same tuple structure applies.
        # Matches v8DetectionLoss internal extraction pattern.
        s_feats = preds[1] if isinstance(preds, tuple) else preds
        t_feats = t_preds[1] if isinstance(t_preds, tuple) else t_preds

        # MSE distillation loss across all 3 detection scales
        kd_loss = torch.tensor(0.0, device=self.device)
        for s_feat, t_feat in zip(s_feats, t_feats):
            # FIX 3: Spatial adaptation — if H/W differ after pruning,
            # resize student feature map to match teacher before MSE.
            if s_feat.shape[-2:] != t_feat.shape[-2:]:
                s_feat = F.interpolate(
                    s_feat, size=t_feat.shape[-2:],
                    mode="bilinear", align_corners=False,
                )
            kd_loss = kd_loss + F.mse_loss(s_feat, t_feat)

        # FIX 4: Convex combination — bounded gradient scale.
        # Old: student_loss + 0.5 * kd_loss  (additive, blows up gradients)
        # New: (1 - α) * student_loss + α * kd_loss  (convex, sum of weights = 1)
        alpha: float = self._kd_alpha
        total_loss = (1.0 - alpha) * student_loss + alpha * kd_loss

        return total_loss, loss_items


print(f"[DEBUG] PrunedTrainer defined — starting training")

model.train(
    data=DATA_YAML,
    trainer=PrunedTrainer,
    epochs=TRAIN_EPOCHS,
    imgsz=TRAIN_IMGSZ,
    batch=TRAIN_BATCH,
    lr0=TRAIN_LR0,
    exist_ok=True,
    warmup_epochs=0,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.3,
    degrees=5.0,
    perspective=0.0003,
    device=0,
    project=OUTPUT_PROJECT,
    name=OUTPUT_NAME,
)

In [ ]:
import os
from ultralytics import YOLO

BEST_PT = os.path.join(OUTPUT_PROJECT, OUTPUT_NAME, 'weights', 'best.pt')
print(f"[DEBUG] Loading: {BEST_PT}")
pruned_model = YOLO(BEST_PT)

metrics_test = pruned_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=TRAIN_IMGSZ,
    batch=32,
    conf=0.001,
    iou=0.6,
    device=0,
    save_json=True,
)

print(f"[TEST] mAP50:    {metrics_test.box.map50:.4f}")
print(f"[TEST] mAP50-95: {metrics_test.box.map:.4f}")
print(f"[TEST] Precision: {metrics_test.box.mp:.4f}")
print(f"[TEST] Recall:    {metrics_test.box.mr:.4f}")

In [ ]:
from ultralytics import YOLO

orig_model = YOLO('yolov8n.pt')
pruned_model = YOLO(BEST_PT)

print("=== ORIGINAL YOLOv8n ===")
orig_info = orig_model.info()
print(f"  Layers: {orig_info[0]} | Params: {orig_info[1]:,} | GFLOPs: {orig_info[3]:.1f}")

print("\n=== PRUNED + KD MODEL ===")
pruned_info = pruned_model.info()
print(f"  Layers: {pruned_info[0]} | Params: {pruned_info[1]:,} | GFLOPs: {pruned_info[3]:.1f}")

param_reduction: float = (1 - pruned_info[1] / orig_info[1]) * 100
flops_reduction: float = (1 - pruned_info[3] / orig_info[3]) * 100
print(f"\n[DELTA] Params: -{param_reduction:.1f}% | GFLOPs: -{flops_reduction:.1f}%")

In [ ]:
from ultralytics import YOLO

export_model = YOLO(BEST_PT)

onnx_path = export_model.export(
    format='onnx',
    imgsz=TRAIN_IMGSZ,
    dynamic=False,
    simplify=True,
    opset=12,
)

print(f"[DEBUG] ONNX exported: {onnx_path}")

import os
onnx_size_mb: float = os.path.getsize(onnx_path) / (1024 * 1024)
print(f"[DEBUG] ONNX size: {onnx_size_mb:.1f} MB")

In [ ]:
from ultralytics import YOLO

val_model = YOLO(BEST_PT)

metrics_val = val_model.val(
    data=DATA_YAML,
    split='val',
    imgsz=TRAIN_IMGSZ,
    batch=32,
    conf=0.001,
    iou=0.6,
    device=0,
    save_json=True,
)

print(f"[VAL] mAP50:    {metrics_val.box.map50:.4f}")
print(f"[VAL] mAP50-95: {metrics_val.box.map:.4f}")
print(f"[VAL] Precision: {metrics_val.box.mp:.4f}")
print(f"[VAL] Recall:    {metrics_val.box.mr:.4f}")